# Automated hAChE peptide-screening pipeline: Protenix + modular docking + MM/GBSA preparation

This Colab notebook is designed as a **replicable screening platform** for peptide candidates against human acetylcholinesterase (hAChE). It supports three docking modes:

```python
DOCKING_MODE = "lightdock"   # LightDock only
DOCKING_MODE = "haddock"     # HADDOCK3 only
DOCKING_MODE = "both"        # run both and integrate the scores
```

The critical change in this version is that the peptide is not represented by a single arbitrary structure. The notebook builds a **peptide conformational ensemble** using two sources:

1. **Context-aware bound conformers** extracted from the best Protenix co-folding models.
2. **Idealized unbound conformers** generated as helix, extended/sheet, polyproline-II and optional random backbone states.

This is important because peptide docking is highly sensitive to the starting peptide conformation. Protenix provides receptor-context sampling, while idealized conformers give a fallback and reduce the risk of over-depending on a single folded pose.

The workflow is:

```text
Peptide CSV
   ↓
Sequence curation and descriptors
   ↓
hAChE reference preparation and PAS/Trp286 mapping
   ↓
Protenix co-folding hAChE–peptide, optionally with soft PAS pocket guidance
   ↓
Geometric scoring of Protenix models: PAS distance, Trp286 distance, contacts
   ↓
Peptide ensemble generation for docking
   ↓
LightDock and/or HADDOCK3 docking with PAS restraints
   ↓
Integrated normalized ranking
   ↓
Optional AmberTools/MMGBSA preparation for top candidates
```

Important scientific rule used here: **Trp286/PAS proximity is a ranking feature, not a hard acceptance rule.** The script avoids selecting a model only because it touches Trp286.

## 0. Colab runtime and installation

Use `Runtime > Change runtime type > GPU` for Protenix. For quick testing, use 1–3 peptides, few Protenix seeds, small docking sampling and `DOCKING_MODE = "lightdock"` or `"haddock"`. For a more robust ranking, use `"both"` and increase sampling.

Note: current Colab runtimes may use Python 3.12. LightDock and HADDOCK3 can both depend on ProDy, which often cannot build there. In Python 3.12, the install cell skips those docking engines and the notebook automatically continues with the available modules. Protenix is installed into an isolated `/content/protenix_env` environment so its heavy ML dependencies do not disturb Colab's notebook kernel. Use a Python 3.10/3.11 environment if you specifically need LightDock or HADDOCK3.

In [ ]:

%%bash
#@title Install core dependencies
set -eu
export DEBIAN_FRONTEND=noninteractive

# Colab's r2u source can emit a harmless Sources warning during apt update.
apt-get update -qq || true
apt-get install -y -qq wget git curl gawk zip build-essential ninja-build
# Keep/restore Colab-compatible pins. This also repairs runtimes where a previous
# failed install upgraded numpy, pandas, protobuf or setuptools too far.
python -m pip -q install 'numpy<2.1' 'pandas==2.2.2' 'protobuf<6.0dev,>=5.26.1' 'setuptools<82' wheel packaging 'jedi>=0.16' virtualenv

# Core packages required by the notebook. These use the numpy/pandas already provided by Colab.
python -m pip -q install biopython pdb-tools py3Dmol PeptideBuilder

set +e
# Optional engines. Keep these optional so one incompatible dependency does not break the notebook.
# Set INSTALL_PROTENIX=true only if you will run RUN_PROTENIX=True later.
INSTALL_PROTENIX=false

PY_MM=$(python - <<'PY'
import sys
print(f'{sys.version_info.major}.{sys.version_info.minor}')
PY
)
if [ "$PY_MM" = "3.12" ]; then
  echo "WARNING: Skipping LightDock and HADDOCK3 on Python 3.12 because their ProDy dependency often cannot build in Colab."
  echo "         Use Protenix/idealized conformer scoring here, or run docking engines in a Python 3.10/3.11 environment."
else
  python -m pip -q install 'prody<3' lightdock==0.9.2post1 || echo "WARNING: LightDock install failed; LightDock cells will be skipped."
  python -m pip -q install 'prody<3' haddock3 || echo "WARNING: HADDOCK3 install failed; HADDOCK cells will be skipped."
fi

# Protenix can be heavy and may alter ML package versions, so install it in an
# isolated virtual environment instead of the Colab notebook kernel.
PROTENIX_ENV=/content/protenix_env
if [ "$INSTALL_PROTENIX" = true ]; then
  rm -rf "$PROTENIX_ENV"
  python -m virtualenv "$PROTENIX_ENV"
  "$PROTENIX_ENV/bin/python" -m pip -q install --upgrade pip setuptools wheel ninja
  "$PROTENIX_ENV/bin/python" -m pip install protenix || echo "WARNING: Protenix install failed; RUN_PROTENIX=True will not work until fixed."
else
  echo "Protenix installation skipped. Set INSTALL_PROTENIX=true in this cell if you plan to run RUN_PROTENIX=True."
fi

# Optional local minimization for peptide conformers. If this fails, leave minimize_peptides_with_openmm=False.
python -m pip -q install openmm pdbfixer || echo "WARNING: OpenMM/PDBFixer install failed; peptide minimization remains optional."

# Final repair pass for the notebook kernel after optional installs.
python -m pip -q install 'numpy<2.1' 'pandas==2.2.2' 'protobuf<6.0dev,>=5.26.1' 'setuptools<82' 'jedi>=0.16'

python - <<'PY'
import importlib.metadata as md
import importlib.util, shutil, sys
from pathlib import Path
protenix_bin = Path('/content/protenix_env/bin/protenix')
checks = {
    'pandas': importlib.util.find_spec('pandas') is not None,
    'Bio': importlib.util.find_spec('Bio') is not None,
    'PeptideBuilder': importlib.util.find_spec('PeptideBuilder') is not None,
    'lightdock3_setup.py': shutil.which('lightdock3_setup.py') is not None,
    'haddock3': shutil.which('haddock3') is not None,
    'protenix': protenix_bin.exists() or shutil.which('protenix') is not None,
}
print('Python:', sys.version.split()[0])
for name, ok in checks.items():
    print(f'{name:20s}:', 'OK' if ok else 'missing')
for pkg in ['numpy', 'pandas', 'protobuf', 'setuptools']:
    try:
        print(f'{pkg:20s}:', md.version(pkg))
    except md.PackageNotFoundError:
        print(f'{pkg:20s}: missing')
if protenix_bin.exists():
    print('protenix executable:', protenix_bin)
PY

In [ ]:

#@title Global configuration
from pathlib import Path
import os, re, json, glob, math, shutil, subprocess, textwrap, random
import pandas as pd
import numpy as np

ROOT = Path('/content/hache_peptide_screening')
RAW = ROOT / 'raw'
PROTENIX_IN = ROOT / 'protenix_inputs'
PROTENIX_OUT = ROOT / 'protenix_outputs'
PEP_STRUCT = ROOT / 'peptide_structures'
SCORES = ROOT / 'scores'
LIGHTDOCK_DIR = ROOT / 'lightdock'
HADDOCK_DIR = ROOT / 'haddock'
AMBER_DIR = ROOT / 'amber_mmgbsa'
for p in [ROOT, RAW, PROTENIX_IN, PROTENIX_OUT, PEP_STRUCT, SCORES, LIGHTDOCK_DIR, HADDOCK_DIR, AMBER_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    # Main user switch: "lightdock", "haddock", or "both"
    'docking_mode': 'both',

    # hAChE reference. 4PQE is apo hAChE; 4EY7 contains donepezil-bound hAChE.
    'target_pdb_id': '4PQE',
    'target_chain': 'A',

    # PAS/hotspot definition for hAChE. Trp82 is not used here; it is the BChE logic.
    # Verify numbering after receptor preparation.
    'main_hotspot_pdbnum': 286,       # hAChE Trp286
    'pas_residues_pdbnum': [72, 74, 124, 286, 341],
    'extra_gorge_residues_pdbnum': [337, 338, 447],

    # Protenix settings
    'use_protenix_pocket_constraint': True,
    'protenix_max_distance_A': 8.0,
    'seeds': [11, 22, 33],
    'protenix_num_samples': 3,
    'protenix_model_name': 'protenix-mini',  # change if your Protenix install expects another name
    'use_msa': False,

    # Peptide ensemble settings for docking
    # mixed = Protenix-bound conformers + idealized conformers
    # protenix_bound = only extracted Protenix peptide conformers
    # idealized = only generated idealized conformers
    'peptide_structure_mode': 'mixed',
    'n_protenix_bound_conformers': 3,
    'idealized_conformers': ['helix', 'sheet', 'ppii', 'extended'],
    'n_random_backbone_conformers': 4,
    'random_seed_for_peptide_builder': 123,
    'minimize_peptides_with_openmm': False,
    'openmm_minimize_max_iter': 250,

    # LightDock settings. Increase for final runs.
    'lightdock_swarms': 20,
    'lightdock_glowworms': 40,
    'lightdock_steps': 50,
    # For energy-like scoring functions such as DFIRE/FastDFIRE, lower is normally better.
    # Change to True if you use a scoring function where higher is better.
    'lightdock_higher_is_better': False,

    # HADDOCK3 settings. Increase sampling for final runs.
    'haddock_ncores': 2,
    'haddock_sampling': 200,
    'haddock_seletop': 50,
    'haddock_cluster_cutoff': 5,
    'haddock_top_models_per_cluster': 4,

    # Final ranking weights. Scores absent because a mode was skipped are automatically ignored.
    'weights': {
        'protenix_site_score_norm': 0.25,
        'lightdock_score_norm': 0.25,
        'haddock_score_norm': 0.25,
        'mmgbsa_score_norm': 0.25,
    },
}

DOCKING_MODE = CONFIG['docking_mode'].lower()
assert DOCKING_MODE in {'lightdock', 'haddock', 'both'}
random.seed(CONFIG['random_seed_for_peptide_builder'])
np.random.seed(CONFIG['random_seed_for_peptide_builder'])

# Runtime availability checks. Some Colab Python versions cannot install every
# optional engine, so keep the available branches running instead of crashing late.
TOOL_STATUS = {
    'lightdock': shutil.which('lightdock3_setup.py') is not None and shutil.which('lightdock3.py') is not None,
    'haddock': shutil.which('haddock3') is not None,
    'haddock_restraints': shutil.which('haddock3-restraints') is not None,
    'protenix': Path('/content/protenix_env/bin/protenix').exists() or shutil.which('protenix') is not None,
}
PROTENIX_CMD = str(Path('/content/protenix_env/bin/protenix')) if Path('/content/protenix_env/bin/protenix').exists() else shutil.which('protenix')

requested_mode = DOCKING_MODE
if requested_mode in {'lightdock', 'both'} and not TOOL_STATUS['lightdock']:
    print('WARNING: LightDock commands were not found; LightDock docking will be skipped.')
    DOCKING_MODE = 'haddock' if requested_mode == 'both' and TOOL_STATUS['haddock'] else 'none'

if DOCKING_MODE in {'haddock', 'both'} and not TOOL_STATUS['haddock']:
    print('WARNING: HADDOCK3 command was not found; HADDOCK3 docking will be skipped.')
    DOCKING_MODE = 'lightdock' if DOCKING_MODE == 'both' and TOOL_STATUS['lightdock'] else 'none'

if requested_mode != DOCKING_MODE:
    print(f'Effective docking mode changed from {requested_mode!r} to {DOCKING_MODE!r}.')
if not TOOL_STATUS['protenix']:
    print('WARNING: Protenix command was not found. Leave RUN_PROTENIX=False or fix the Protenix install first.')
else:
    print('Protenix command:', PROTENIX_CMD)

print('Workdir:', ROOT)
print('Docking mode:', DOCKING_MODE)

## 1. Load peptide candidates

Upload or create a CSV with at least:

```text
peptide_id,sequence
pep1,LGWVSKGKLL
```

Optional metadata columns such as `cterm_amidated`, `source`, `notes` are preserved. C-terminal amidation is recorded as metadata. Full chemical capping of termini is not automatically imposed in this base notebook; if termini are essential for a final candidate, validate that step manually in AmberTools/Rosetta or another preparation workflow.

In [ ]:

#@title Create example peptide CSV or replace it with your own upload
example = pd.DataFrame({
    'peptide_id': ['pep_demo_LGWVSKGKLL'],
    'sequence': ['LGWVSKGKLL'],
    'cterm_amidated': [True]
})
peptides_csv = ROOT / 'peptides.csv'
example.to_csv(peptides_csv, index=False)
print(f'Peptide CSV: {peptides_csv}')
example

In [ ]:

#@title Basic sequence curation and descriptors
AA = set('ACDEFGHIKLMNPQRSTVWY')
HYDRO = {
    'A': 1.8, 'C': 2.5, 'D': -3.5, 'E': -3.5, 'F': 2.8, 'G': -0.4,
    'H': -3.2, 'I': 4.5, 'K': -3.9, 'L': 3.8, 'M': 1.9, 'N': -3.5,
    'P': -1.6, 'Q': -3.5, 'R': -4.5, 'S': -0.8, 'T': -0.7,
    'V': 4.2, 'W': -0.9, 'Y': -1.3
}

def clean_seq(seq):
    return re.sub('[^A-Z]', '', str(seq).upper())

def net_charge_rough(seq):
    return sum(seq.count(x) for x in 'KR') - sum(seq.count(x) for x in 'DE') + 0.1 * seq.count('H')

def mean_hydropathy(seq):
    return float(np.mean([HYDRO[a] for a in seq])) if seq else np.nan

peptides = pd.read_csv(peptides_csv)
peptides['sequence'] = peptides['sequence'].map(clean_seq)
peptides['valid_natural_aa'] = peptides['sequence'].map(lambda s: bool(s) and set(s).issubset(AA))
peptides['length'] = peptides['sequence'].str.len()
peptides['net_charge_rough'] = peptides['sequence'].map(net_charge_rough)
peptides['hydropathy_mean'] = peptides['sequence'].map(mean_hydropathy)
peptides['passes_basic_filter'] = peptides['valid_natural_aa'] & peptides['length'].between(5, 40)

peptides.to_csv(SCORES / '01_curated_peptides.csv', index=False)
peptides

## 2. Download and prepare hAChE reference

The reference chain is used for three things:

1. Obtaining a clean receptor PDB for docking.
2. Mapping PDB residue numbers to sequence positions for Protenix input.
3. Creating PAS restraints for LightDock/HADDOCK3.

Always verify the residue numbering if you change the PDB ID or chain.

In [ ]:

#@title Download hAChE structure and extract clean receptor chain
from Bio.PDB import PDBParser, MMCIFParser, PDBIO, Select
from Bio.PDB.Polypeptide import protein_letters_3to1_extended
import urllib.request

three_to_one = {k.upper(): v for k, v in protein_letters_3to1_extended.items()}
one_to_three = {
    'A':'ALA','C':'CYS','D':'ASP','E':'GLU','F':'PHE','G':'GLY','H':'HIS','I':'ILE','K':'LYS','L':'LEU','M':'MET',
    'N':'ASN','P':'PRO','Q':'GLN','R':'ARG','S':'SER','T':'THR','V':'VAL','W':'TRP','Y':'TYR'
}

def run_cmd(cmd, cwd=None, allow_fail=False):
    cmd = [str(x) for x in cmd]
    print('RUN:', ' '.join(cmd))
    try:
        result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    except FileNotFoundError as e:
        print(f'Command not found: {cmd[0]}')
        if not allow_fail:
            raise RuntimeError(f'Command not found: {cmd[0]}') from e
        return subprocess.CompletedProcess(cmd, 127, '', str(e))
    if result.stdout:
        print(result.stdout[-3000:])
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr[-4000:])
        if not allow_fail:
            raise RuntimeError(f'Command failed: {cmd}')
    return result

def download_pdb(pdb_id, out_dir=RAW):
    pdb_id = pdb_id.upper()
    out_path = out_dir / f'{pdb_id}.pdb'
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    urllib.request.urlretrieve(url, out_path)
    return out_path

class CleanChainSelect(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id
    def accept_chain(self, chain):
        return chain.id == self.chain_id
    def accept_residue(self, residue):
        return residue.id[0] == ' ' and residue.get_resname().upper() in three_to_one
    def accept_atom(self, atom):
        return atom.element != 'H'

def extract_chain_sequence_and_mapping(pdb_path, chain_id):
    parser = PDBParser(QUIET=True)
    s = parser.get_structure('target', str(pdb_path))
    chain = s[0][chain_id]
    seq = []
    pdbnum_to_seqpos = {}
    seqpos_to_pdbnum = {}
    for res in chain:
        if res.id[0] != ' ':
            continue
        resname = res.get_resname().upper()
        if resname not in three_to_one:
            continue
        seq.append(three_to_one[resname])
        seqpos = len(seq)
        pdbnum = res.id[1]
        pdbnum_to_seqpos[pdbnum] = seqpos
        seqpos_to_pdbnum[seqpos] = pdbnum
    return ''.join(seq), pdbnum_to_seqpos, seqpos_to_pdbnum

pdb_path = download_pdb(CONFIG['target_pdb_id'])
hache_seq, pdbnum_to_seqpos, seqpos_to_pdbnum = extract_chain_sequence_and_mapping(pdb_path, CONFIG['target_chain'])

# Save clean receptor for docking with original PDB residue numbering.
receptor_clean = RAW / f"{CONFIG['target_pdb_id']}_{CONFIG['target_chain']}_hAChE_clean.pdb"
s = PDBParser(QUIET=True).get_structure('target', str(pdb_path))
io = PDBIO(); io.set_structure(s)
io.save(str(receptor_clean), CleanChainSelect(CONFIG['target_chain']))

print('Reference PDB:', pdb_path)
print('Clean receptor:', receptor_clean)
print('hAChE sequence length from selected chain:', len(hache_seq))
print('PAS PDB → Protenix sequence-position mapping:')
for r in CONFIG['pas_residues_pdbnum']:
    print(f'  PDB residue {r} -> seqpos {pdbnum_to_seqpos.get(r)}')

missing = [r for r in CONFIG['pas_residues_pdbnum'] if r not in pdbnum_to_seqpos]
if missing:
    print('WARNING: these PAS residues were not found in the selected chain:', missing)

## 3. Protenix co-folding inputs

Protenix is used to generate receptor-context peptide poses. The PAS `pocket` restraint is soft guidance, not a hard acceptance rule. A useful validation is to run a second no-restraint batch and compare whether the peptide still tends to sample the PAS.

In [ ]:

#@title Generate Protenix JSON input files

def make_protenix_json(peptide_id, peptide_seq, hache_seq, pas_seq_positions, out_json):
    job = {
        'name': peptide_id,
        'sequences': [
            {'proteinChain': {'sequence': hache_seq, 'count': 1, 'id': ['A']}},
            {'proteinChain': {'sequence': peptide_seq, 'count': 1, 'id': ['B']}},
        ]
    }
    if CONFIG['use_protenix_pocket_constraint']:
        job['constraint'] = {
            'pocket': {
                'binder_chain': {'entity': 2, 'copy': 1},
                'contact_residues': [
                    {'entity': 1, 'copy': 1, 'position': int(pos)}
                    for pos in pas_seq_positions if pos is not None
                ],
                'max_distance': CONFIG['protenix_max_distance_A']
            }
        }
    with open(out_json, 'w') as f:
        json.dump(job, f, indent=2)
    return out_json

pas_seq_positions = [pdbnum_to_seqpos.get(x) for x in CONFIG['pas_residues_pdbnum']]
protenix_jobs = []
for _, row in peptides[peptides['passes_basic_filter']].iterrows():
    peptide_id, seq = row['peptide_id'], row['sequence']
    out_json = PROTENIX_IN / f'{peptide_id}.json'
    make_protenix_json(peptide_id, seq, hache_seq, pas_seq_positions, out_json)
    protenix_jobs.append({'peptide_id': peptide_id, 'sequence': seq, 'json': str(out_json)})

pd.DataFrame(protenix_jobs).to_csv(SCORES / '02_protenix_jobs.csv', index=False)
pd.DataFrame(protenix_jobs)

In [ ]:

#@title Run Protenix co-folding
# This cell can be skipped if you already have Protenix output files and place them under PROTENIX_OUT.
# If the installed Protenix CLI differs, edit the command construction below.

RUN_PROTENIX = False #@param {type:"boolean"}

if RUN_PROTENIX and not TOOL_STATUS.get('protenix'):
    raise RuntimeError('RUN_PROTENIX=True but the protenix command is missing. Re-run the install cell or leave RUN_PROTENIX=False.')

if RUN_PROTENIX:
    for job in protenix_jobs:
        peptide_id = job['peptide_id']
        for seed in CONFIG['seeds']:
            out_dir = PROTENIX_OUT / peptide_id / f'seed_{seed}'
            out_dir.mkdir(parents=True, exist_ok=True)
            cmd = [
                PROTENIX_CMD, 'pred',
                '-i', job['json'],
                '-o', str(out_dir),
                '--seeds', str(seed),
                '--num_samples', str(CONFIG['protenix_num_samples']),
            ]
            if CONFIG.get('protenix_model_name'):
                cmd += ['--model_name', CONFIG['protenix_model_name']]
            if not CONFIG.get('use_msa', False):
                cmd += ['--use_msa', 'false']
            run_cmd(cmd)
else:
    print('RUN_PROTENIX=False. Place existing .cif/.pdb Protenix models in:', PROTENIX_OUT)

## 4. Geometric scoring of Protenix models

The script scores all detected Protenix structures using:

- minimum heavy-atom distance to hAChE Trp286,
- minimum heavy-atom distance to the PAS set,
- number of peptide–PAS contacts within 5 Å,
- number of peptide–Trp286 contacts within 5 Å.

This score is used to select a **small ensemble of plausible bound peptide conformers**, not to make the final decision alone.

In [ ]:

#@title Structure reading and geometric scoring functions
from Bio.PDB import MMCIFParser, PDBParser, PDBIO, Select

class OnlyChain(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id
    def accept_chain(self, chain):
        return chain.id == self.chain_id
    def accept_atom(self, atom):
        return atom.element != 'H'


def find_structure_files(base_dir):
    files = glob.glob(str(base_dir / '**' / '*.cif'), recursive=True)
    files += glob.glob(str(base_dir / '**' / '*.pdb'), recursive=True)
    return sorted(files)


def load_structure_any(path):
    path = str(path)
    if path.lower().endswith('.cif'):
        return MMCIFParser(QUIET=True).get_structure(Path(path).stem, path)
    return PDBParser(QUIET=True).get_structure(Path(path).stem, path)


def heavy_atom_coords_residue(chain, resseq):
    coords = []
    for res in chain:
        if res.id[0] != ' ':
            continue
        if res.id[1] == int(resseq):
            for atom in res:
                if atom.element != 'H':
                    coords.append(atom.coord)
    return np.array(coords, dtype=float) if coords else np.empty((0,3))


def heavy_atom_coords_chain(chain):
    coords = []
    for res in chain:
        if res.id[0] != ' ':
            continue
        for atom in res:
            if atom.element != 'H':
                coords.append(atom.coord)
    return np.array(coords, dtype=float) if coords else np.empty((0,3))


def min_distance(A, B):
    if A.size == 0 or B.size == 0:
        return np.nan
    # efficient enough for these systems
    diff = A[:, None, :] - B[None, :, :]
    return float(np.sqrt((diff * diff).sum(axis=2)).min())


def count_contacts(A, B, cutoff=5.0):
    if A.size == 0 or B.size == 0:
        return 0
    diff = A[:, None, :] - B[None, :, :]
    d2 = (diff * diff).sum(axis=2)
    return int((d2 <= cutoff * cutoff).sum())


def score_complex_geometry(struct_file, peptide_id, receptor_chain='A', peptide_chain='B'):
    s = load_structure_any(struct_file)
    model = next(s.get_models())
    if receptor_chain not in model or peptide_chain not in model:
        return None
    rec = model[receptor_chain]
    pep = model[peptide_chain]
    pep_coords = heavy_atom_coords_chain(pep)

    trp_seqpos = pdbnum_to_seqpos.get(CONFIG['main_hotspot_pdbnum'])
    trp_coords = heavy_atom_coords_residue(rec, trp_seqpos) if trp_seqpos else np.empty((0,3))

    pas_coords_list = []
    for pdbnum in CONFIG['pas_residues_pdbnum']:
        seqpos = pdbnum_to_seqpos.get(pdbnum)
        if seqpos:
            cc = heavy_atom_coords_residue(rec, seqpos)
            if cc.size:
                pas_coords_list.append(cc)
    pas_coords = np.vstack(pas_coords_list) if pas_coords_list else np.empty((0,3))

    return {
        'peptide_id': peptide_id,
        'model_file': str(struct_file),
        'dmin_trp286_A': min_distance(pep_coords, trp_coords),
        'dmin_pas_A': min_distance(pep_coords, pas_coords),
        'contacts_trp286_5A': count_contacts(pep_coords, trp_coords, 5.0),
        'contacts_pas_5A': count_contacts(pep_coords, pas_coords, 5.0),
    }

rows = []
for job in protenix_jobs:
    peptide_id = job['peptide_id']
    for sf in find_structure_files(PROTENIX_OUT / peptide_id):
        r = score_complex_geometry(sf, peptide_id)
        if r:
            rows.append(r)
geom = pd.DataFrame(rows)

def norm_0_100(series, higher_is_better=True):
    x = pd.to_numeric(series, errors='coerce')
    out = pd.Series(np.nan, index=series.index, dtype=float)
    if x.notna().sum() == 0:
        return out
    xmin, xmax = np.nanmin(x), np.nanmax(x)
    if xmax == xmin:
        out.loc[x.notna()] = 50.0
        return out
    z = (x - xmin) / (xmax - xmin)
    if not higher_is_better:
        z = 1 - z
    return 100 * z

if len(geom):
    geom['dmin_trp286_norm'] = norm_0_100(geom['dmin_trp286_A'], higher_is_better=False)
    geom['dmin_pas_norm'] = norm_0_100(geom['dmin_pas_A'], higher_is_better=False)
    geom['contacts_pas_norm'] = norm_0_100(geom['contacts_pas_5A'], higher_is_better=True)
    geom['contacts_trp286_norm'] = norm_0_100(geom['contacts_trp286_5A'], higher_is_better=True)
    geom['protenix_site_score_norm'] = (
        0.30 * geom['dmin_trp286_norm'] +
        0.30 * geom['dmin_pas_norm'] +
        0.25 * geom['contacts_pas_norm'] +
        0.15 * geom['contacts_trp286_norm']
    )
else:
    print('No Protenix model files found yet.')

geom.to_csv(SCORES / '03_protenix_geometric_scores.csv', index=False)
geom.head(20)

## 5. Peptide conformational ensemble for docking

This is the most important upgrade. Each peptide receives an ensemble containing, depending on `peptide_structure_mode`:

- top Protenix-bound peptide conformations extracted from context-aware hAChE–peptide models;
- idealized conformers generated from the sequence: α-helix, β/sheet-like, polyproline-II and extended;
- optional random backbone conformers sampled around common Ramachandran regions.

HADDOCK3 can use this as a multi-model peptide PDB. LightDock is run once per conformer and then summarized per peptide.

In [ ]:

#@title Build peptide conformer ensemble
import PeptideBuilder
from Bio.PDB import PDBParser, PDBIO

# Backbone angle presets. PeptideBuilder expects one phi and psi_im1 value per residue after the first.
ANGLE_PRESETS = {
    'helix':    (-57, -47),
    'sheet':    (-135, 135),
    'ppii':     (-75, 145),
    'extended': (-120, 140),
}
RAMA_CENTERS = [(-57, -47), (-135, 135), (-75, 145), (-120, 140)]


def save_structure_standard_chain(structure, out_pdb, chain_id='B', renumber=True):
    # Keep first model and first chain, assign chain ID, optionally renumber residues 1..N.
    model = next(structure.get_models())
    chains = list(model.get_chains())
    if len(chains) != 1:
        # keep only the first chain if needed
        for ch in chains[1:]:
            model.detach_child(ch.id)
    chain = list(model.get_chains())[0]
    chain.id = chain_id
    if renumber:
        residues = [r for r in chain if r.id[0] == ' ']
        for i, res in enumerate(residues, start=1):
            res.id = (' ', i, ' ')
    io = PDBIO(); io.set_structure(structure)
    io.save(str(out_pdb))
    return out_pdb


def build_ideal_peptide(sequence, kind, out_pdb, chain_id='B'):
    n = len(sequence)
    if n < 2:
        raise ValueError('Peptide too short for conformer generation')
    if kind == 'random':
        center_phi, center_psi = random.choice(RAMA_CENTERS)
        phi = [float(np.random.normal(center_phi, 18)) for _ in range(n-1)]
        psi = [float(np.random.normal(center_psi, 18)) for _ in range(n-1)]
        structure = PeptideBuilder.make_structure(sequence, phi, psi)
    else:
        phi0, psi0 = ANGLE_PRESETS[kind]
        structure = PeptideBuilder.make_structure(sequence, [phi0]*(n-1), [psi0]*(n-1))
    return save_structure_standard_chain(structure, out_pdb, chain_id=chain_id, renumber=True)


def minimize_peptide_openmm(in_pdb, out_pdb):
    try:
        from openmm.app import PDBFile, Modeller, ForceField, Simulation, NoCutoff, HBonds
        from openmm import LangevinMiddleIntegrator, unit
        pdb = PDBFile(str(in_pdb))
        ff = ForceField('amber14-all.xml', 'implicit/gbn2.xml')
        modeller = Modeller(pdb.topology, pdb.positions)
        modeller.addHydrogens(ff, pH=7.4)
        system = ff.createSystem(modeller.topology, nonbondedMethod=NoCutoff, constraints=HBonds)
        integrator = LangevinMiddleIntegrator(300*unit.kelvin, 1/unit.picosecond, 0.004*unit.picoseconds)
        sim = Simulation(modeller.topology, system, integrator)
        sim.context.setPositions(modeller.positions)
        sim.minimizeEnergy(maxIterations=CONFIG['openmm_minimize_max_iter'])
        state = sim.context.getState(getPositions=True)
        with open(out_pdb, 'w') as f:
            PDBFile.writeFile(modeller.topology, state.getPositions(), f)
        return out_pdb
    except Exception as e:
        print(f'OpenMM minimization failed for {in_pdb}: {e}')
        shutil.copyfile(in_pdb, out_pdb)
        return out_pdb


def save_chain_from_structure(struct_file, chain_id, out_pdb, renumber=True):
    s = load_structure_any(struct_file)
    model = next(s.get_models())
    if chain_id not in model:
        raise ValueError(f'Chain {chain_id} not found in {struct_file}')
    # Create a structure containing only chain_id
    io = PDBIO(); io.set_structure(s)
    tmp = Path(out_pdb).with_suffix('.tmp.pdb')
    io.save(str(tmp), OnlyChain(chain_id))
    s2 = PDBParser(QUIET=True).get_structure('pep', str(tmp))
    save_structure_standard_chain(s2, out_pdb, chain_id='B', renumber=renumber)
    tmp.unlink(missing_ok=True)
    return out_pdb


def make_multimodel_ensemble(pdb_files, out_pdb):
    with open(out_pdb, 'w') as out:
        model_i = 1
        for pf in pdb_files:
            out.write(f'MODEL     {model_i:4d}\n')
            with open(pf) as f:
                for line in f:
                    if line.startswith(('ATOM', 'HETATM')):
                        out.write(line)
            out.write('ENDMDL\n')
            model_i += 1
    return out_pdb

peptide_conformer_rows = []

for _, prow in peptides[peptides['passes_basic_filter']].iterrows():
    peptide_id, seq = prow['peptide_id'], prow['sequence']
    pdir = PEP_STRUCT / peptide_id
    pdir.mkdir(parents=True, exist_ok=True)
    conformer_files = []

    # A. Protenix-bound conformers
    if CONFIG['peptide_structure_mode'] in {'mixed', 'protenix_bound'} and len(geom):
        top_models = (geom[geom['peptide_id'] == peptide_id]
                      .sort_values('protenix_site_score_norm', ascending=False)
                      .head(CONFIG['n_protenix_bound_conformers']))
        for j, row in enumerate(top_models.itertuples(), start=1):
            out_pdb = pdir / f'{peptide_id}_protenix_bound_{j:02d}.pdb'
            try:
                save_chain_from_structure(row.model_file, 'B', out_pdb, renumber=True)
                conformer_files.append(out_pdb)
                peptide_conformer_rows.append({'peptide_id': peptide_id, 'conformer_id': out_pdb.stem, 'source': 'protenix_bound', 'pdb': str(out_pdb)})
            except Exception as e:
                print('Could not extract Protenix peptide:', peptide_id, row.model_file, e)

    # B. Idealized conformers
    if CONFIG['peptide_structure_mode'] in {'mixed', 'idealized'}:
        for kind in CONFIG['idealized_conformers']:
            raw_pdb = pdir / f'{peptide_id}_{kind}.raw.pdb'
            out_pdb = pdir / f'{peptide_id}_{kind}.pdb'
            build_ideal_peptide(seq, kind, raw_pdb, chain_id='B')
            if CONFIG['minimize_peptides_with_openmm']:
                minimize_peptide_openmm(raw_pdb, out_pdb)
            else:
                shutil.copyfile(raw_pdb, out_pdb)
            conformer_files.append(out_pdb)
            peptide_conformer_rows.append({'peptide_id': peptide_id, 'conformer_id': out_pdb.stem, 'source': f'idealized_{kind}', 'pdb': str(out_pdb)})

        for k in range(CONFIG['n_random_backbone_conformers']):
            raw_pdb = pdir / f'{peptide_id}_random_{k+1:02d}.raw.pdb'
            out_pdb = pdir / f'{peptide_id}_random_{k+1:02d}.pdb'
            build_ideal_peptide(seq, 'random', raw_pdb, chain_id='B')
            if CONFIG['minimize_peptides_with_openmm']:
                minimize_peptide_openmm(raw_pdb, out_pdb)
            else:
                shutil.copyfile(raw_pdb, out_pdb)
            conformer_files.append(out_pdb)
            peptide_conformer_rows.append({'peptide_id': peptide_id, 'conformer_id': out_pdb.stem, 'source': 'idealized_random', 'pdb': str(out_pdb)})

    if conformer_files:
        ensemble_pdb = pdir / f'{peptide_id}_ensemble.pdb'
        make_multimodel_ensemble(conformer_files, ensemble_pdb)
        print(peptide_id, 'conformers:', len(conformer_files), 'ensemble:', ensemble_pdb)
    else:
        print('WARNING: no conformers generated for', peptide_id)

peptide_conformers = pd.DataFrame(peptide_conformer_rows)
peptide_conformers.to_csv(SCORES / '04_peptide_conformers.csv', index=False)
peptide_conformers.head(20)

## 6A. LightDock docking mode

LightDock is run separately for each peptide conformer. This is slower than one single docking per peptide, but scientifically safer because the starting conformation matters. The final LightDock score per peptide is the best conformer-level score.

In [ ]:

#@title Prepare LightDock restraints

def make_lightdock_restraints(out_file):
    # Receptor restraints use the receptor PDB numbering.
    # P = passive residues, blank/default = active. We keep Trp286 active and nearby PAS passive.
    with open(out_file, 'w') as f:
        for pdbnum in CONFIG['pas_residues_pdbnum']:
            aa1 = None
            if pdbnum in pdbnum_to_seqpos:
                aa1 = hache_seq[pdbnum_to_seqpos[pdbnum]-1]
            aa3 = one_to_three.get(aa1, 'UNK')
            passive_flag = '' if pdbnum == CONFIG['main_hotspot_pdbnum'] else ' P'
            f.write(f'R {CONFIG["target_chain"]}.{aa3}.{pdbnum}{passive_flag}\n')
        # Ligand side: allow the whole peptide as passive, because the binding residue is unknown.
        f.write('# Ligand peptide residues are left unrestricted/passive by default.\n')
    return out_file

print('LightDock restraints preview:')
tmp_rst = SCORES / 'lightdock_restraints_preview.list'
make_lightdock_restraints(tmp_rst)
print(tmp_rst.read_text())

In [ ]:

#@title Run LightDock for each peptide conformer
RUN_LIGHTDOCK = False #@param {type:"boolean"}

lightdock_rows = []
if DOCKING_MODE in {'lightdock', 'both'}:
    if RUN_LIGHTDOCK:
        if peptide_conformers.empty:
            raise RuntimeError('No peptide conformers available. Run the peptide ensemble cell first.')
        for item in peptide_conformers.itertuples():
            peptide_id = item.peptide_id
            conformer_id = item.conformer_id
            lig_pdb = Path(item.pdb)
            ddir = LIGHTDOCK_DIR / peptide_id / conformer_id
            ddir.mkdir(parents=True, exist_ok=True)
            rec_pdb = ddir / 'receptor.pdb'
            pep_pdb = ddir / 'peptide.pdb'
            shutil.copyfile(receptor_clean, rec_pdb)
            shutil.copyfile(lig_pdb, pep_pdb)
            rst = ddir / 'restraints.list'
            make_lightdock_restraints(rst)

            # Clean previous run files
            for pat in ['lightdock*', 'setup.json', 'init', 'swarm_*', 'rank_by_scoring.list', 'filtered']:
                for x in glob.glob(str(ddir / pat)):
                    if os.path.isdir(x): shutil.rmtree(x)
                    else: os.remove(x)

            run_cmd([
                'lightdock3_setup.py', str(rec_pdb), str(pep_pdb),
                '--noxt', '--noh', '--now', '-anm', '-rst', str(rst),
                '-s', str(CONFIG['lightdock_swarms']),
                '-g', str(CONFIG['lightdock_glowworms'])
            ], cwd=ddir)
            run_cmd(['lightdock3.py', 'setup.json', str(CONFIG['lightdock_steps']), '-s', 'fastdfire'], cwd=ddir)
            run_cmd(['lgd_rank.py', str(CONFIG['lightdock_swarms']), 'fastdfire'], cwd=ddir, allow_fail=True)
    else:
        print('RUN_LIGHTDOCK=False. Existing LightDock results will be parsed if present.')
else:
    print('Skipping LightDock because DOCKING_MODE =', DOCKING_MODE)

In [ ]:

#@title Parse LightDock scores

def read_lightdock_rank(rank_file, peptide_id, conformer_id):
    rows = []
    rank_file = Path(rank_file)
    if not rank_file.exists():
        return rows
    with open(rank_file) as f:
        for line in f:
            if not line.strip() or line.startswith('#'):
                continue
            parts = line.split()
            nums = []
            for p in parts:
                try:
                    nums.append(float(p))
                except Exception:
                    pass
            if nums:
                rows.append({
                    'peptide_id': peptide_id,
                    'conformer_id': conformer_id,
                    'lightdock_score_raw': nums[-1],
                    'rank_line': line.strip(),
                    'rank_file': str(rank_file)
                })
    return rows

ld_rows = []
if DOCKING_MODE in {'lightdock', 'both'} and not peptide_conformers.empty:
    for item in peptide_conformers.itertuples():
        rank_file = LIGHTDOCK_DIR / item.peptide_id / item.conformer_id / 'rank_by_scoring.list'
        ld_rows.extend(read_lightdock_rank(rank_file, item.peptide_id, item.conformer_id))

lightdock_scores = pd.DataFrame(ld_rows)
if len(lightdock_scores):
    # Best row per conformer is first in rank file; then best conformer per peptide based on score direction.
    lightdock_scores['row_order'] = lightdock_scores.groupby(['peptide_id','conformer_id']).cumcount()
    top_per_conformer = lightdock_scores[lightdock_scores['row_order'] == 0].copy()
    top_per_conformer = top_per_conformer.sort_values(
        ['peptide_id', 'lightdock_score_raw'],
        ascending=[True, not CONFIG['lightdock_higher_is_better']]
    )
    lightdock_summary = top_per_conformer.groupby('peptide_id', as_index=False).head(1)
    lightdock_summary['lightdock_score_norm'] = norm_0_100(
        lightdock_summary['lightdock_score_raw'],
        higher_is_better=CONFIG['lightdock_higher_is_better']
    )
else:
    lightdock_summary = pd.DataFrame(columns=['peptide_id','conformer_id','lightdock_score_raw','lightdock_score_norm','rank_file'])

lightdock_scores.to_csv(SCORES / '05_lightdock_all_scores.csv', index=False)
lightdock_summary.to_csv(SCORES / '06_lightdock_summary.csv', index=False)
lightdock_summary.head(20)

## 6B. HADDOCK3 docking mode

HADDOCK3 uses one receptor PDB plus a multi-model peptide ensemble PDB. The restraint strategy is:

- hAChE PAS residues are active on the receptor side, with Trp286 included.
- all peptide residues are passive, because the exact peptide residue mediating binding is unknown.

This guides docking to the PAS while still allowing different peptide orientations.

In [ ]:

#@title Prepare HADDOCK3 files and configuration

def write_act_pass(active_residues, passive_residues, out_file):
    with open(out_file, 'w') as f:
        f.write(' '.join(str(x) for x in active_residues) + '\n')
        f.write(' '.join(str(x) for x in passive_residues) + '\n')
    return out_file


def make_haddock_config(peptide_id, receptor_pdb, peptide_ensemble_pdb, ambig_tbl, out_cfg):
    run_dir = HADDOCK_DIR / peptide_id / 'run'
    cfg = f'''
# hAChE peptide docking with HADDOCK3
run_dir = "{run_dir}"
mode = "local"
ncores = {CONFIG['haddock_ncores']}
postprocess = true
clean = true

molecules = [
  "{receptor_pdb}",
  "{peptide_ensemble_pdb}"
]

[topoaa]

[rigidbody]
ambig_fname = "{ambig_tbl}"
sampling = {CONFIG['haddock_sampling']}

[caprieval]

[flexref]
tolerance = 5
ambig_fname = "{ambig_tbl}"
# For final runs, consider increasing mdsteps_cool1, mdsteps_rigid, mdsteps_cool2 and mdsteps_cool3.

[caprieval]

[emref]
tolerance = 5
ambig_fname = "{ambig_tbl}"

[caprieval]

[seletop]
select = {CONFIG['haddock_seletop']}

[caprieval]

[rmsdmatrix]

[clustrmsd]
clust_cutoff = {CONFIG['haddock_cluster_cutoff']}
plot_matrix = true

[caprieval]

[seletopclusts]
top_models = {CONFIG['haddock_top_models_per_cluster']}

[caprieval]
'''
    Path(out_cfg).write_text(textwrap.dedent(cfg).strip() + '\n')
    return out_cfg

haddock_jobs = []
if DOCKING_MODE in {'haddock', 'both'}:
    for _, prow in peptides[peptides['passes_basic_filter']].iterrows():
        peptide_id, seq = prow['peptide_id'], prow['sequence']
        pdir = HADDOCK_DIR / peptide_id
        pdir.mkdir(parents=True, exist_ok=True)
        rec = pdir / 'receptor_hache.pdb'
        pep_ens = pdir / 'peptide_ensemble.pdb'
        shutil.copyfile(receptor_clean, rec)
        src_ens = PEP_STRUCT / peptide_id / f'{peptide_id}_ensemble.pdb'
        if not src_ens.exists():
            print('Missing ensemble, skipping HADDOCK job:', peptide_id)
            continue
        shutil.copyfile(src_ens, pep_ens)

        rec_act = pdir / 'receptor.act-pass'
        pep_pass = pdir / 'peptide.act-pass'
        # Active receptor residues: PAS set. Passive receptor residues omitted because peptide has no active residues.
        write_act_pass(CONFIG['pas_residues_pdbnum'], [], rec_act)
        write_act_pass([], list(range(1, len(seq)+1)), pep_pass)
        ambig = pdir / 'ambig.tbl'
        cfg = pdir / 'haddock3_hache_peptide.cfg'

        # Generate AIR restraints using haddock3-restraints if available.
        result = None
        if TOOL_STATUS.get('haddock_restraints'):
            cmd = ['haddock3-restraints', 'active_passive_to_ambig', str(rec_act), str(pep_pass)]
            result = subprocess.run(cmd, text=True, capture_output=True)
        if result is not None and result.returncode == 0 and result.stdout.strip():
            ambig.write_text(result.stdout)
        else:
            # Fallback: generic ambiguous restraints from receptor PAS residues to any peptide residue CA/CB.
            # This is less elegant than haddock3-restraints but keeps the notebook inspectable.
            lines = []
            pep_sel = ' or '.join([f'(segid B and resid {i})' for i in range(1, len(seq)+1)])
            for r in CONFIG['pas_residues_pdbnum']:
                lines.append(f'assign (segid A and resid {r}) ({pep_sel}) 3.0 2.0 8.0')
            ambig.write_text('\n'.join(lines) + '\n')
            print('WARNING: haddock3-restraints unavailable or failed; wrote fallback ambig.tbl for', peptide_id)
            if result is not None and result.stderr:
                print(result.stderr[-1000:])
        if TOOL_STATUS.get('haddock_restraints'):
            run_cmd(['haddock3-restraints', 'validate_tbl', str(ambig), '--silent'], allow_fail=True)
        make_haddock_config(peptide_id, rec, pep_ens, ambig, cfg)
        haddock_jobs.append({'peptide_id': peptide_id, 'cfg': str(cfg), 'run_dir': str(pdir / 'run'), 'ambig_tbl': str(ambig)})
else:
    print('Skipping HADDOCK3 because DOCKING_MODE =', DOCKING_MODE)

pd.DataFrame(haddock_jobs).to_csv(SCORES / '07_haddock_jobs.csv', index=False)
pd.DataFrame(haddock_jobs)

In [ ]:

#@title Run HADDOCK3
RUN_HADDOCK = False #@param {type:"boolean"}

if DOCKING_MODE in {'haddock', 'both'}:
    if RUN_HADDOCK:
        for job in haddock_jobs:
            cfg = job['cfg']
            run_cmd(['haddock3', cfg], cwd=Path(cfg).parent)
    else:
        print('RUN_HADDOCK=False. Existing HADDOCK3 run directories will be parsed if present.')
else:
    print('Skipping HADDOCK3 because DOCKING_MODE =', DOCKING_MODE)

In [ ]:

#@title Parse HADDOCK3 scores

def find_last_haddock_score_table(run_dir):
    run_dir = Path(run_dir)
    # Prefer final clustered CAPRI table, then final single-structure table.
    candidates = sorted(run_dir.glob('**/capri_clt.tsv')) + sorted(run_dir.glob('**/capri_ss.tsv'))
    if not candidates:
        return None
    return candidates[-1]


def read_haddock_score(run_dir, peptide_id):
    table = find_last_haddock_score_table(run_dir)
    if table is None:
        return None
    try:
        df = pd.read_csv(table, sep='\t')
    except Exception:
        return None
    if df.empty:
        return None
    score_cols = [c for c in df.columns if 'score' in c.lower()]
    if not score_cols:
        return None
    score_col = score_cols[0]
    df[score_col] = pd.to_numeric(df[score_col], errors='coerce')
    best = df.sort_values(score_col, ascending=True).head(1).copy()  # lower/more negative HADDOCK score is better
    row = best.iloc[0].to_dict()
    return {
        'peptide_id': peptide_id,
        'haddock_score_raw': row.get(score_col),
        'haddock_score_col': score_col,
        'haddock_table': str(table),
        'haddock_row_json': json.dumps(row, default=str)
    }

haddock_rows = []
if DOCKING_MODE in {'haddock', 'both'}:
    for job in haddock_jobs:
        r = read_haddock_score(job['run_dir'], job['peptide_id'])
        if r:
            haddock_rows.append(r)

haddock_summary = pd.DataFrame(haddock_rows)
if len(haddock_summary):
    haddock_summary['haddock_score_norm'] = norm_0_100(haddock_summary['haddock_score_raw'], higher_is_better=False)
else:
    haddock_summary = pd.DataFrame(columns=['peptide_id','haddock_score_raw','haddock_score_norm','haddock_table'])

haddock_summary.to_csv(SCORES / '08_haddock_summary.csv', index=False)
haddock_summary.head(20)

## 7. Optional AmberTools/MMGBSA preparation

For a screening notebook, MM/GBSA should be reserved for the best few candidates after Protenix+docking. This cell installs AmberTools. Topology generation and residue decomposition must be validated carefully, especially for protonation states, termini, amidation and histidines.

In [ ]:

%%bash
#@title Install AmberTools with micromamba (optional, heavy)
set -e
INSTALL_AMBERTOOLS=false
if [ "$INSTALL_AMBERTOOLS" = true ]; then
  cd /content
  if [ ! -x /content/bin/micromamba ]; then
    curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
  fi
  /content/bin/micromamba create -y -p /content/ambertools_env -c conda-forge ambertools=24.8 parmed openmm pdbfixer
  /content/bin/micromamba run -p /content/ambertools_env MMPBSA.py -h | head -20
else
  echo "AmberTools installation skipped. Edit INSTALL_AMBERTOOLS=true when needed."
fi

In [ ]:

#@title Write MMPBSA.py input template
mmpbsa_in = AMBER_DIR / 'mmpbsa_decomp.in'
mmpbsa_in.write_text('''&general
  startframe=1, endframe=1, interval=1,
  verbose=1,
/
&gb
  igb=5, saltcon=0.150,
/
&decomp
  idecomp=1,
  dec_verbose=1,
/
''')
print(mmpbsa_in.read_text())
print('Expected command after validated topology/snapshot preparation:')
print('/content/bin/micromamba run -p /content/ambertools_env MMPBSA.py -O -i mmpbsa_decomp.in -cp complex.prmtop -rp receptor.prmtop -lp peptide.prmtop -y snapshots.nc -o FINAL_RESULTS_MMPBSA.dat -do FINAL_DECOMP_MMPBSA.dat')

## 8. Integrated ranking

The ranking keeps all raw values and normalizes available scores to 0–100. If one module is skipped, its weight is automatically removed and the remaining weights are rescaled.

Interpretation:

- `protenix_site_score_norm`: PAS-compatible co-folded geometry.
- `lightdock_score_norm`: best LightDock conformer score if LightDock was run.
- `haddock_score_norm`: best HADDOCK3 score/cluster score if HADDOCK3 was run.
- `mmgbsa_score_norm`: placeholder until real MM/GBSA values are added.

Final selection for synthesis should also include visual inspection, chemical feasibility, solubility/stability and experimental prioritization.

In [ ]:

#@title Build final ranking
base = peptides[['peptide_id','sequence','length','net_charge_rough','hydropathy_mean','passes_basic_filter']].copy()

if len(geom):
    prot_summary = (geom.sort_values('protenix_site_score_norm', ascending=False)
                    .groupby('peptide_id', as_index=False)
                    .head(1)[[
                        'peptide_id','model_file','dmin_trp286_A','dmin_pas_A',
                        'contacts_trp286_5A','contacts_pas_5A','protenix_site_score_norm'
                    ]])
else:
    prot_summary = pd.DataFrame(columns=['peptide_id','model_file','dmin_trp286_A','dmin_pas_A','contacts_trp286_5A','contacts_pas_5A','protenix_site_score_norm'])

rank = base.merge(prot_summary, on='peptide_id', how='left')
rank = rank.merge(lightdock_summary[['peptide_id','conformer_id','lightdock_score_raw','lightdock_score_norm']], on='peptide_id', how='left')
rank = rank.rename(columns={'conformer_id': 'best_lightdock_conformer'})
rank = rank.merge(haddock_summary[['peptide_id','haddock_score_raw','haddock_score_norm']], on='peptide_id', how='left')

# Placeholder for real MM/GBSA values. Add a CSV with columns peptide_id, mmgbsa_score_raw if available.
rank['mmgbsa_score_raw'] = np.nan
rank['mmgbsa_score_norm'] = np.nan

weights = CONFIG['weights'].copy()
score_cols = list(weights.keys())
final_scores = []
used_weights_rows = []
for _, row in rank.iterrows():
    available = {c: weights[c] for c in score_cols if c in rank.columns and pd.notna(row[c])}
    if not available:
        final_scores.append(np.nan)
        used_weights_rows.append('{}')
        continue
    wsum = sum(available.values())
    score = sum((available[c]/wsum) * row[c] for c in available)
    final_scores.append(score)
    used_weights_rows.append(json.dumps({c: available[c]/wsum for c in available}))
rank['Final_score'] = final_scores
rank['used_normalized_weights'] = used_weights_rows
rank = rank.sort_values('Final_score', ascending=False)

rank.to_csv(SCORES / '09_final_ranking.csv', index=False)
rank

## 9. Export results

Download the ZIP and inspect:

- `01_curated_peptides.csv`
- `03_protenix_geometric_scores.csv`
- `04_peptide_conformers.csv`
- `06_lightdock_summary.csv`
- `08_haddock_summary.csv`
- `09_final_ranking.csv`

For final candidates, inspect the 3D pose manually before synthesis and validate MM/GBSA setup separately.

In [ ]:

%%bash
#@title Zip results
cd /content
zip -qr hache_peptide_screening_results.zip hache_peptide_screening/scores hache_peptide_screening/protenix_inputs hache_peptide_screening/peptide_structures || true
ls -lh /content/hache_peptide_screening_results.zip || true